# Sequence Extrapolation

**Track:** Learning | **Function:** Pattern recognition & mathematical generalization

**What it measures:** Can the model learn the pattern in a number sequence and predict the next terms?

Each item presents a number sequence generated by a hidden mathematical rule. The model must identify the pattern and predict the next value.

**Pre/post learning format:**
- *Baseline:* Predict the next number directly from the sequence
- *Post-learning:* Analyze the sequence pattern, write notes, then predict with notes as context

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import random
import string
import re

print('Available models:', list(kbench.llms.keys()))
print('Default model:', kbench.llm)

In [ ]:
def strip_thinking(text: str) -> str:
    """Remove <think>...</think> blocks from reasoning model output."""
    if '</think>' in text:
        return text.split('</think>', 1)[1].strip()
    return text.strip()


def extract_short_answer(response: str) -> str:
    """Extract a short answer from model response (last short line)."""
    text = strip_thinking(response)
    text = re.sub(r'[`*_]', '', text)
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    if not lines:
        return ''
    # Prefer last short line (skip explanations)
    for line in reversed(lines):
        clean = line.strip('"\'\'\u201c\u201d').rstrip('.!?')
        if len(clean) <= 30:
            return clean.lower().strip()
    return lines[-1].strip('"\'\'\u201c\u201d').rstrip('.!?').lower().strip()

In [ ]:
# === Sequence rule generators ===

def seq_linear(seed, n):
    """a*i + b"""
    rng = random.Random(seed)
    a = rng.randint(2, 9)
    b = rng.randint(-5, 15)
    return [a*i + b for i in range(n)], f'{a}*i + {b}'

def seq_quadratic(seed, n):
    """a*i^2 + b*i + c"""
    rng = random.Random(seed)
    a = rng.randint(1, 4)
    b = rng.randint(-3, 5)
    c = rng.randint(0, 10)
    return [a*i*i + b*i + c for i in range(n)], f'{a}*i^2 + {b}*i + {c}'

def seq_geometric_add(seed, n):
    """f(i) = f(i-1)*r + d, f(0) = start"""
    rng = random.Random(seed)
    start = rng.randint(1, 5)
    r = rng.choice([2, 3])
    d = rng.randint(-3, 5)
    seq = [start]
    for _ in range(n-1):
        seq.append(seq[-1]*r + d)
    return seq, f'f(i)=f(i-1)*{r}+{d}, f(0)={start}'

def seq_alternating(seed, n):
    """Even positions: +a, Odd positions: +b"""
    rng = random.Random(seed)
    start = rng.randint(1, 10)
    a = rng.randint(2, 7)
    b = rng.randint(2, 7)
    while b == a:
        b = rng.randint(2, 7)
    seq = [start]
    for i in range(1, n):
        seq.append(seq[-1] + (a if i % 2 == 0 else b))
    return seq, f'alt +{a}/+{b} from {start}'

def seq_fibonacci_variant(seed, n):
    """f(i) = f(i-1) + f(i-2) + c"""
    rng = random.Random(seed)
    a, b = rng.randint(1, 5), rng.randint(1, 5)
    c = rng.randint(0, 3)
    seq = [a, b]
    for _ in range(n-2):
        seq.append(seq[-1] + seq[-2] + c)
    return seq, f'fib+{c} from ({a},{b})'

def seq_conditional(seed, n):
    """if prev is even: +a, if prev is odd: *2-b"""
    rng = random.Random(seed)
    start = rng.randint(1, 8)
    a = rng.randint(3, 7)
    b = rng.randint(1, 4)
    seq = [start]
    for _ in range(n-1):
        if seq[-1] % 2 == 0:
            seq.append(seq[-1] + a)
        else:
            seq.append(seq[-1] * 2 - b)
    return seq, f'even:+{a}, odd:*2-{b}, start={start}'

FAMILIES = {
    'easy': [seq_linear, seq_alternating],
    'medium': [seq_quadratic, seq_fibonacci_variant],
    'hard': [seq_geometric_add, seq_conditional],
}

# Sanity check
s, desc = seq_conditional(42, 8)
print(f'Example ({desc}): {s}')

In [ ]:
SHOWN = {'easy': 8, 'medium': 7, 'hard': 6}  # terms shown
TOTAL_LEN = 10  # generate 10 terms, show N, test on next
SEEDS = 20
rows = []
tid = 0

for diff in ['easy', 'medium', 'hard']:
    n_shown = SHOWN[diff]
    for gen_fn in FAMILIES[diff]:
        for seed in range(SEEDS):
            seq, rule_desc = gen_fn(seed * 100, TOTAL_LEN)
            shown = seq[:n_shown]
            target = seq[n_shown]  # predict the next term
            seq_text = ', '.join(str(x) for x in shown)
            rows.append({
                'task_id': tid, 'seed': seed, 'difficulty': diff,
                'family': gen_fn.__name__, 'rule_desc': rule_desc,
                'n_shown': n_shown, 'sequence_text': seq_text,
                'expected': str(target),
            })
            tid += 1

dataset = pd.DataFrame(rows)
print(f'Dataset: {len(dataset)} items')
print(dataset.groupby(['difficulty','family']).size().to_string())

In [ ]:
BASELINE_P = '''Here is a number sequence that follows a hidden mathematical rule:

{sequence}

What is the next number in this sequence?

Reply with ONLY the number. No explanation.'''

STUDY_P = '''Here is a number sequence that follows a hidden mathematical rule:

{sequence}

Analyze this sequence carefully.
- Compute the differences between consecutive terms.
- Look for patterns in those differences.
- Check if the rule depends on position, parity, or previous terms.
- Write down the EXACT formula or rule that generates this sequence.
- Verify your rule produces every term shown.'''

APPLY_P = '''You previously analyzed a number sequence and wrote these notes:

--- YOUR NOTES ---
{notes}
--- END NOTES ---

The sequence was: {sequence}

Using your analysis, what is the next number?

Reply with ONLY the number. No explanation.'''

def extract_number(response):
    text = strip_thinking(response)
    text = re.sub(r'[`*_]', '', text)
    # Find all numbers in the response
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    # Prefer last line that is just a number
    for line in reversed(lines):
        clean = line.strip('.!?, ')
        if re.match(r'^-?\d+$', clean):
            return clean
    # Fallback: last number in text
    nums = re.findall(r'-?\d+', text)
    return nums[-1] if nums else ''

## Task 1: Baseline (predict directly from sequence)

In [ ]:
@kbench.task(name='sequence_baseline',
             description='Predict next term in a number sequence directly')
def sequence_baseline(llm, sequence_text:str, expected:str,
                      difficulty:str, seed:int, family:str,
                      rule_desc:str, n_shown:int) -> bool:
    resp = llm.prompt(BASELINE_P.format(sequence=sequence_text))
    return extract_number(resp) == expected

baseline_runs = sequence_baseline.evaluate(
    llm=[kbench.llm], evaluation_data=dataset.set_index('task_id'),
    n_jobs=4, timeout=3600, max_attempts=1)
baseline_df = baseline_runs.as_dataframe()
print(f'Baseline done. Accuracy: {baseline_df["result"].mean():.1%}' if 'result' in baseline_df.columns else 'Baseline done.')

## Task 2: Post-learning (analyze pattern + predict with notes)

In [ ]:
@kbench.task(name='sequence_learning',
             description='Analyze sequence pattern, generate notes, then predict with notes')
def sequence_learning(llm, sequence_text:str, expected:str,
                      difficulty:str, seed:int, family:str,
                      rule_desc:str, n_shown:int) -> bool:
    notes = strip_thinking(llm.prompt(STUDY_P.format(sequence=sequence_text)))
    resp = llm.prompt(APPLY_P.format(notes=notes, sequence=sequence_text))
    return extract_number(resp) == expected

learning_runs = sequence_learning.evaluate(
    llm=[kbench.llm], evaluation_data=dataset.set_index('task_id'),
    n_jobs=4, timeout=3600, max_attempts=1)
learning_df = learning_runs.as_dataframe()
print(f'Learning done. Accuracy: {learning_df["result"].mean():.1%}' if 'result' in learning_df.columns else 'Learning done.')

## Results

In [ ]:
# === Combine results ===
# Debug: inspect results structure
print('Baseline columns:', baseline_df.columns.tolist())
print('Baseline index name:', baseline_df.index.name)
print('Baseline shape:', baseline_df.shape)
print('First rows:')
print(baseline_df.head(3))
print()

# The results DataFrames from .as_dataframe() already contain the evaluation data columns.
# Use them directly instead of trying to join back to the original dataset.
baseline_acc = baseline_df['result'].mean() if 'result' in baseline_df.columns else 0.0
learning_acc = learning_df['result'].mean() if 'result' in learning_df.columns else 0.0
learning_gain = learning_acc - baseline_acc
room = 1.0 - baseline_acc
efficiency = learning_gain / room if room > 0 else 0.0

print('=' * 60)
print('SEQUENCE EXTRAPOLATION — LEARNING RESULTS')
print('=' * 60)
print(f'Baseline accuracy:      {baseline_acc:.1%}')
print(f'Post-learning accuracy: {learning_acc:.1%}')
print(f'Learning gain:          {learning_gain:+.1%}')
print(f'Learning efficiency:    {efficiency:.1%} (of possible improvement)')
print()

# Per-difficulty breakdown using the results DataFrames directly
print('By Difficulty:')
print('-' * 60)
diff_col = 'difficulty'
for diff in ['easy', 'medium', 'hard']:
    b_sub = baseline_df[baseline_df[diff_col] == diff] if diff_col in baseline_df.columns else pd.DataFrame()
    l_sub = learning_df[learning_df[diff_col] == diff] if diff_col in learning_df.columns else pd.DataFrame()
    if len(b_sub) == 0 and len(l_sub) == 0:
        continue
    b = b_sub['result'].mean() if len(b_sub) > 0 and 'result' in b_sub.columns else 0.0
    l = l_sub['result'].mean() if len(l_sub) > 0 and 'result' in l_sub.columns else 0.0
    g = l - b
    n = max(len(b_sub), len(l_sub))
    print(f'  {diff:8s}  baseline={b:.1%}  learned={l:.1%}  gain={g:+.1%}  (n={n})')

# Learning impact analysis
print()
if 'result' in baseline_df.columns and 'result' in learning_df.columns:
    # Merge on index to compare per-item
    merged = baseline_df[['result']].rename(columns={'result':'baseline'}).join(
        learning_df[['result']].rename(columns={'result':'learning'}),
        how='inner'
    )
    helped = ((~merged['baseline']) & merged['learning']).sum()
    hurt = (merged['baseline'] & (~merged['learning'])).sum()
    print(f'Learning helped (wrong->right): {helped}')
    print(f'Learning hurt (right->wrong):   {hurt}')
    print(f'Net learning impact:            {helped - hurt:+d} items')


In [ ]:
!pip install -q matplotlib
import matplotlib.pyplot as plt

difficulties = ['easy', 'medium', 'hard']
diff_col = 'difficulty'
baseline_scores = []
learning_scores = []
diff_labels = []

for d in difficulties:
    b_sub = baseline_df[baseline_df[diff_col] == d] if diff_col in baseline_df.columns else pd.DataFrame()
    l_sub = learning_df[learning_df[diff_col] == d] if diff_col in learning_df.columns else pd.DataFrame()
    if len(b_sub) == 0:
        continue
    baseline_scores.append(b_sub['result'].mean())
    learning_scores.append(l_sub['result'].mean() if len(l_sub) > 0 else 0.0)
    diff_labels.append(d)

x = range(len(diff_labels))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
ax1.bar([i - width/2 for i in x], baseline_scores, width, label='Baseline', color='#4C72B0')
ax1.bar([i + width/2 for i in x], learning_scores, width, label='Post-Learning', color='#55A868')
ax1.set_xlabel('Difficulty')
ax1.set_ylabel('Accuracy')
ax1.set_title('Baseline vs Post-Learning Accuracy')
ax1.set_xticks(list(x))
ax1.set_xticklabels(diff_labels)
ax1.legend()
ax1.set_ylim(0, 1.05)

ax2 = axes[1]
gains = [l - b for b, l in zip(baseline_scores, learning_scores)]
colors = ['#55A868' if g >= 0 else '#C44E52' for g in gains]
ax2.bar(diff_labels, gains, color=colors)
ax2.set_xlabel('Difficulty')
ax2.set_ylabel('Learning Gain')
ax2.set_title('Learning Gain by Difficulty')
ax2.axhline(y=0, color='gray', linestyle='--', linewidth=0.8)

plt.tight_layout()
plt.savefig('sequence_extrapolation_results.png', dpi=150, bbox_inches='tight')
plt.show()
